# 🎯 Module 03: LLM APIs & Model Providers

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 1 — Foundations  
> **Module:** 03 — LLM APIs & Model Providers  
> **Planned Date:** July 6, 2026  
> **Level:** Beginner → Advanced  
> **Prerequisites:** Python, env vars, Modules 01–02  
> **Previous:** 02 — LLM Fundamentals  
> **Next:** 04 — Prompt Engineering

---

Now it's time to actually **talk to a model** from Python.

In this module we'll cover:

- What an **LLM API** is
- **Model providers** and **SDKs**
- Setting up the **environment**
- Your **first LLM request**
- **Message roles** (`system`, `user`, `assistant`, `tool`)
- **Requests & responses**
- **Streaming**
- **Sync vs async**
- **Error handling**
- **Retries, backoff, idempotency**
- **Rate limits & concurrency**
- **Cost & token usage**
- **Security & privacy**
- **Testing** API integrations
- Building a **reusable LLM client**

## 📚 Learning Objectives

By the end of this module you should be able to:

1. Explain **APIs, SDKs, and model providers**
2. Configure a provider **without hardcoding credentials**
3. Send a **successful model request**
4. Understand **message roles** and **response structure**
5. Inspect **usage metadata** where available
6. Implement / understand **streaming**
7. Distinguish **sync** from **async** requests
8. Handle **timeouts and provider errors**
9. Understand **retries, backoff, idempotency, rate limits**
10. Estimate **costs** using applicable pricing
11. Protect **credentials and sensitive data**
12. Write **unit tests** using mocked responses
13. Build a **reusable LLM client**

---
## ⚙️ 0. Setup & Imports

This notebook is **safe to run without an API key**. Where a real call would be made, we show the code but guard it behind configuration checks.

In [ ]:
import os
import time
import json
import asyncio
from pathlib import Path
from types import SimpleNamespace

print('✅ Setup ready.')
print('Working directory:', Path.cwd())

# Detect whether we have live credentials configured
HAS_KEY = bool(os.getenv('LLM_API_KEY'))
HAS_MODEL = bool(os.getenv('LLM_MODEL'))
print(f'LLM_API_KEY present: {HAS_KEY}')
print(f'LLM_MODEL   present: {HAS_MODEL}')

if not (HAS_KEY and HAS_MODEL):
    print('\n⚠️ Live calls are disabled. Code cells that would call a real API are')
    print('   wrapped in `if HAS_KEY and HAS_MODEL:` checks and skipped safely.')

---
## 🧠 1. What Is an LLM API?

An **API** (Application Programming Interface) lets one software system talk to another through a defined interface.

An **LLM API** lets an application send input to a language model and get back a generated response.

### Typical request flow

```text
User
  |
  v
Python Application
  |
  v
LLM Provider SDK
  |
  v
HTTP API Request
  |
  v
Model Inference
  |
  v
API Response
  |
  v
Application Processes the Result
```

> ⚠️ **Calling an API performs inference** — it does **not** train or fine-tune the model.

---
## 🔌 2. Providers & SDKs

A **model provider** offers access to models through a hosted API, local deployment, or another interface.

An **SDK** is a library that helps you interact with that interface in your language.

### Common integration approaches

| Approach | Description | When to use |
| --- | --- | --- |
| Provider SDK | Official library | Working with one provider |
| Direct HTTP | Use `requests` / `httpx` | Full control over HTTP |
| OpenAI-compatible API | Compatible request format | Service supports it |
| Local inference server | Model on your machine | Local deployment |

> "OpenAI-compatible" may mean **partial** compatibility — verify endpoints, auth, fields, responses.

### Best practice

- Start with the **official SDK**
- Also learn **HTTP basics** for debugging
- **Isolate** provider-specific code behind a small internal interface

---
## 🛠️ 3. Set Up Your Environment

### Suggested structure

```text
03-LLM-APIs-and-Model-Providers/
├── README.md
├── requirements.txt
├── .env.example
├── .gitignore
├── src/
│   ├── __init__.py
│   └── basic_request.py
├── examples/
│   └── streaming.py
└── tests/
    └── test_helpers.py
```

### Install

```bash
python -m pip install openai python-dotenv pytest
python -m pip freeze > requirements.txt
```

### `.env.example`

```dotenv
LLM_API_KEY=replace_with_your_key
LLM_MODEL=replace_with_a_supported_model_id
LLM_BASE_URL=
```

### `.gitignore`

```gitignore
.env
.venv/
__pycache__/
.pytest_cache/
*.py[cod]
```

> Use the model ID and endpoint **documented by your provider**. Do not assume all providers support the same models or base URL.

In [ ]:
# Inspect what the environment looks like right now
for var in ['LLM_API_KEY', 'LLM_MODEL', 'LLM_BASE_URL']:
    value = os.getenv(var)
    if var == 'LLM_API_KEY' and value:
        # Mask the key — never print the raw value
        masked = value[:4] + '...' + value[-4:] if len(value) > 8 else '(too short)'
        print(f'{var:12s} = {masked}')
    else:
        print(f'{var:12s} = {value!r}')

---
## 🚀 4. Your First LLM API Request

The example below uses the OpenAI Python SDK and its **chat completions** interface. It assumes the provider supports this interface and that the selected model is available to your account.

```python
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ['LLM_API_KEY'],
    base_url=os.getenv('LLM_BASE_URL') or None,
)

response = client.chat.completions.create(
    model=os.environ['LLM_MODEL'],
    messages=[
        {'role': 'system', 'content': 'You are a helpful Python tutor.'},
        {'role': 'user',   'content': 'Explain an API in simple English.'},
    ],
)

print(response.choices[0].message.content or '')
```

### Understand the code

- `load_dotenv()` — loads `.env`
- `OpenAI(...)` — SDK client
- `model` — which model to use
- `messages` — conversation input
- `role` — message purpose
- `create(...)` — sends the request
- `response.choices[0].message.content` — the generated text

> ⚠️ This interface is **not universal** — some providers use different methods, formats, or response objects.

In [ ]:
# Optional: run a real request ONLY if credentials are configured
if HAS_KEY and HAS_MODEL:
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
        load_dotenv()

        client = OpenAI(
            api_key=os.environ['LLM_API_KEY'],
            base_url=os.getenv('LLM_BASE_URL') or None,
        )
        response = client.chat.completions.create(
            model=os.environ['LLM_MODEL'],
            messages=[
                {'role': 'system', 'content': 'You are a helpful Python tutor.'},
                {'role': 'user',   'content': 'Explain an API in one sentence.'},
            ],
        )
        print('Model response:')
        print(response.choices[0].message.content or '')
    except Exception as e:
        print(f'Live call failed: {type(e).__name__}: {e}')
else:
    print('Skipped live call — no LLM_API_KEY / LLM_MODEL configured.')

---
## 🎭 5. Message Roles

Many conversational APIs use **role-based** messages.

| Role | Purpose |
| --- | --- |
| `system` | High-level behavior instructions |
| `developer` | Application-level instructions (where supported) |
| `user` | The user's request |
| `assistant` | Model-generated messages |
| `tool` | Tool results (in tool-calling interfaces) |

> Not every provider supports every role the same way. Follow the API docs.

### Example

```python
messages = [
    {'role': 'system', 'content': 'Explain programming concepts clearly.'},
    {'role': 'user',   'content': 'What is a Python dictionary?'},
]
```

> ⚠️ Message roles are **not a security boundary**. Instructions ≠ authorization. Real permissions must be enforced by the application.

In [ ]:
# A conversation as role-tagged messages
conversation = [
    {'role': 'system',    'content': 'Explain programming concepts clearly and concisely.'},
    {'role': 'user',      'content': 'What is a Python dictionary?'},
    {'role': 'assistant', 'content': 'A dictionary maps keys to values.'},
    {'role': 'user',      'content': 'Give an example.'},
]

for msg in conversation:
    print(f"[{msg['role']:9s}] {msg['content']}")

---
## 📨 6. API Requests & Responses

### A request commonly contains

- The **model**
- **Input messages** or another input format
- **Generation settings**
- Optional **response format / tools**
- Provider-specific fields

### A response may contain

- Generated **content**
- **Response ID**
- **Model metadata**
- **Token usage**
- **Finish reason**
- Provider-specific fields

> ⚠️ **Do not assume every field is always present.**

In [ ]:
# Inspect usage metadata — the code is defensive to handle missing fields
def show_usage(response):
    usage = getattr(response, 'usage', None)
    if usage is None:
        print('No usage metadata available.')
        return
    for attr in ['prompt_tokens', 'completion_tokens', 'total_tokens']:
        val = getattr(usage, attr, None)
        print(f'  {attr:18s} = {val}')

# Example with a mock response object
mock_response = SimpleNamespace(
    usage=SimpleNamespace(prompt_tokens=12, completion_tokens=8, total_tokens=20)
)
show_usage(mock_response)

---
## 🌊 7. Streaming Responses

Without streaming, the app **waits** for the whole response.

With streaming, the API sends **chunks** as generation progresses.

### Why it matters

- Better **perceived responsiveness**
- Users see text **as it's written**
- Great for **chat interfaces**
- Requires handling **partial output** and **interruptions**

```python
stream = client.chat.completions.create(
    model=model, messages=messages, stream=True,
)
for chunk in stream:
    if not chunk.choices: continue
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end='', flush=True)
```

> ⚠️ In production, handle **failures, cancellations, incomplete output, disconnects**. Never treat partial output as validated.

---
## ⚡ 8. Sync vs Async

### Sync
Blocks until the request completes or fails.

### Async
Allows the app to manage **multiple I/O operations** concurrently.

```python
from openai import AsyncOpenAI

async with AsyncOpenAI(...) as client:
    response = await client.chat.completions.create(...)
```

> ⚠️ Async does **not** make a single request faster. It helps with **waiting & concurrency**.

> Respect **provider rate limits** before launching concurrent requests.

In [ ]:
# Illustrate concurrency vs sequential with fake async work
async def fake_call(name, delay):
    await asyncio.sleep(delay)
    return f'{name} done after {delay}s'

async def sequential():
    t0 = time.time()
    for i in range(3):
        await fake_call(f'seq-{i}', 0.3)
    return time.time() - t0

async def concurrent():
    t0 = time.time()
    await asyncio.gather(*(fake_call(f'con-{i}', 0.3) for i in range(3)))
    return time.time() - t0

t_seq = asyncio.run(sequential())
t_con = asyncio.run(concurrent())

print(f'Sequential: {t_seq:.2f}s')
print(f'Concurrent: {t_con:.2f}s')
print('Concurrency helps when waiting on independent I/O.')

---
## 🚨 9. Error Handling

| Failure | Cause | Response |
| --- | --- | --- |
| Authentication | Missing / invalid key | Check creds securely |
| Permission | No access to model | Verify account & model |
| Rate limit | Limits exceeded | Retry with backoff |
| Timeout | Too slow | Timeouts + handle failure |
| Server error | Provider-side | Retry selectively |
| Invalid request | Bad input / settings | Fix the request |
| Network error | Connection issue | Handle temporary failure |

```python
from openai import APIConnectionError, APITimeoutError, APIStatusError, RateLimitError

try:
    response = client.chat.completions.create(...)
except RateLimitError:    print('Rate limited.')
except APITimeoutError:   print('Timed out.')
except APIConnectionError: print('Connection error.')
except APIStatusError as e: print(f'HTTP {e.status_code}')
```

> ⚠️ Avoid stacking **uncontrolled retries** on top of the SDK's own retries.

---
## 🔁 10. Retries, Backoff & Idempotency

### Exponential backoff

```text
Attempt 1 → wait 1s
Attempt 2 → wait 2s
Attempt 3 → wait 4s
```

Use **jitter** to avoid many clients retrying at once.

### ✅ Retry these

- Rate limits
- Connection issues
- Transient server errors

### ❌ Do NOT retry blindly

- Invalid requests
- Invalid credentials
- Unsupported models
- Permanent permission failures

### Idempotency

An operation is **idempotent** if repeating it has the same intended effect.

> A repeated generation may still incur **extra cost** or return a **different result**.

In [ ]:
# Exponential backoff with jitter — implementation sketch
import random

def backoff_delays(max_attempts=4, base=1.0, cap=10.0):
    for attempt in range(max_attempts):
        raw = base * (2 ** attempt)
        jitter = random.uniform(0, base)
        yield min(cap, raw + jitter)

print('Example backoff schedule (seconds):')
for i, delay in enumerate(backoff_delays(), 1):
    print(f'  Attempt {i}: wait {delay:.2f}s')

---
## 🚦 11. Rate Limits & Concurrency

Providers may limit requests by **count**, **tokens**, **tier**, **model**, etc.

### Good practices

- Read the provider's **current** rate-limit docs
- Respect **retry guidance** and relevant headers
- **Limit concurrent requests**
- Use **bounded queues**
- Avoid **retry storms**
- **Track** rate-limit errors
- Batch only when supported
- Design a useful response when the service is unavailable

> ⚠️ **Do not hardcode** a limit from an example. Limits change.

---
## 💰 12. Cost & Token Usage

Cost factors:

- Input tokens
- Output tokens
- Model selection
- Cached input (where supported)
- Image / audio / other multimodal inputs
- Extra tools / services

### Example (illustrative pricing)

- $0.50 / 1M input tokens
- $1.50 / 1M output tokens

For 2,000 input + 500 output tokens:

```text
Input  : 2000 / 1e6 * 0.50 = $0.001
Output :  500 / 1e6 * 1.50 = $0.00075
Total  : $0.00175
```

> ⚠️ **These prices are illustrative only.** Always check the model's current pricing page.

In [ ]:
# Simple cost estimator (user-supplied prices)
def estimate_cost(input_tokens, output_tokens, in_price_per_m, out_price_per_m):
    in_cost  = input_tokens  / 1_000_000 * in_price_per_m
    out_cost = output_tokens / 1_000_000 * out_price_per_m
    return {'input_cost': round(in_cost, 6),
            'output_cost': round(out_cost, 6),
            'total_cost': round(in_cost + out_cost, 6)}

example = estimate_cost(
    input_tokens=2_000, output_tokens=500,
    in_price_per_m=0.50, out_price_per_m=1.50,
)
print(json.dumps(example, indent=2))

---
## 🔐 13. Security & Privacy

### API key security

- Store in **env vars / secret manager**
- **Never commit** keys
- **Never** expose server-side keys in frontend JS
- **Rotate** exposed keys
- **Separate** credentials per environment
- **Restrict** access to production secrets

### Data privacy

- Send only what's **needed**
- Understand **provider data terms**
- Avoid logging **secrets & PII**
- Apply **auth & authorization** before retrieving private data
- Review **legal / org requirements**

### Trust boundaries

> **A model response is not automatically safe or valid.**

- Validate outputs before use in **DB queries, file ops, shell, external actions**
- A prompt is **not** a security boundary
- Permissions must be **enforced by application code**

---
## 🧪 14. Testing LLM API Integrations

Tests should check **your application's** behavior — without needing a live API call for every run.

### Test

- Missing configuration
- Request construction
- Response parsing
- Empty / unexpected responses
- Provider errors
- Timeouts & retry behavior
- Logging that avoids secrets
- Output validation
- Behavior when provider is unavailable

In [ ]:
# A testable helper for safely extracting content from a chat response
def extract_message_content(response):
    choices = getattr(response, 'choices', None)
    if not choices:
        return ''
    message = getattr(choices[0], 'message', None)
    if message is None:
        return ''
    content = getattr(message, 'content', None)
    return content if isinstance(content, str) else ''

# --- mini "unit tests" you'd normally put in tests/test_helpers.py ---
def test_extracts_message_content():
    response = SimpleNamespace(
        choices=[SimpleNamespace(message=SimpleNamespace(content='Hello'))]
    )
    assert extract_message_content(response) == 'Hello'

def test_returns_empty_when_no_choices():
    response = SimpleNamespace(choices=[])
    assert extract_message_content(response) == ''

def test_handles_missing_content():
    response = SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=None))])
    assert extract_message_content(response) == ''

for fn in [test_extracts_message_content, test_returns_empty_when_no_choices, test_handles_missing_content]:
    fn()
    print(f'✅ {fn.__name__} passed')

---
## ⚠️ 15. Common Mistakes

1. Hardcoding an **API key** in source
2. Assuming **every provider** uses the same format
3. Assuming **every model ID** is available to every account
4. **Ignoring timeouts** and transient failures
5. **Retrying** invalid requests repeatedly
6. Sending **unlimited concurrent** requests
7. **Logging** private prompts / responses unnecessarily
8. Treating **streamed partial text** as a completed answer
9. **Ignoring token usage** and cost
10. Testing only with **live API calls**
11. Assuming `2xx` → **correct answer**
12. Letting **model output** perform sensitive actions without validation

---
## 🏗️ 16. Practical Project — Reusable LLM Client

### 🎯 Objective
Build a small Python module that makes model requests through a provider SDK and **separates integration code from application logic**.

### Required features

- Environment-based configuration
- Configurable model ID
- Reusable client / wrapper
- `send_message(prompt) -> str` function
- Configurable **timeout**
- Appropriate **error handling**
- **Token usage** reporting when available
- Unit tests with **mocked responses**
- Clear **README** instructions

### Suggested structure

```text
llm-client/
├── README.md
├── requirements.txt
├── .env.example
├── .gitignore
├── src/
│   ├── __init__.py
│   ├── config.py
│   ├── client.py
│   └── helpers.py
└── tests/
    ├── test_helpers.py
    └── test_client.py
```

### Milestones

1. One successful API request
2. Move config into `config.py`
3. Add `send_message(prompt)` function
4. Add timeout + error handling
5. Add tests that **don't need a live key**
6. Record **token usage** when available
7. Document setup, limitations, usage

### Stretch goals

- Streaming
- Async requests
- Bounded concurrency
- Structured logging
- Second provider behind a common interface
- Simple cost estimator

In [ ]:
# Reference implementation of a reusable client — safe to run without live creds
class LLMClient:
    """Thin wrapper around a chat-completions-compatible SDK.

    Only constructs the underlying SDK client when credentials are present.
    Designed so you can swap providers behind a stable interface."""

    def __init__(self, api_key=None, model=None, base_url=None, timeout=30.0, max_retries=2):
        self.api_key = api_key or os.getenv('LLM_API_KEY')
        self.model   = model   or os.getenv('LLM_MODEL')
        self.base_url = base_url or os.getenv('LLM_BASE_URL') or None
        self.timeout = timeout
        self.max_retries = max_retries
        self._client = None

    def _ensure_client(self):
        if self._client is not None:
            return self._client
        if not self.api_key:
            raise RuntimeError('LLM_API_KEY not configured.')
        from openai import OpenAI
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
            timeout=self.timeout,
            max_retries=self.max_retries,
        )
        return self._client

    def send_message(self, prompt, system=None):
        client = self._ensure_client()
        messages = []
        if system:
            messages.append({'role': 'system', 'content': system})
        messages.append({'role': 'user', 'content': prompt})
        response = client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        return extract_message_content(response)


client = LLMClient()
print('LLMClient initialized.')
print('Model:', client.model)
print('Base URL:', client.base_url)
print('Timeout:', client.timeout, 's')

if HAS_KEY and HAS_MODEL:
    try:
        print('\nLive request:')
        print(client.send_message('Explain an API in one sentence.', system='Be concise.'))
    except Exception as e:
        print(f'Live call failed: {type(e).__name__}: {e}')
else:
    print('\nSkipping live call (no credentials). Client is ready to use when configured.')

---
## 🎤 17. Interview Questions

**Q1. What is an LLM API?**  
An interface that lets an application submit inputs to a language model and receive outputs.

**Q2. SDK vs API?**  
The API defines **how** a service is used. An SDK provides **library helpers** for that service.

**Q3. Why env vars for API keys?**  
They separate sensitive config from source. Production may use a dedicated **secret manager**.

**Q4. What is streaming?**  
Delivering output **incrementally** instead of waiting for the full response.

**Q5. When is async useful?**  
For managing **concurrent I/O** efficiently when multiple requests can be in flight.

**Q6. What is exponential backoff?**  
A retry strategy that increases the wait between attempts, often with jitter and a max cap.

**Q7. Why not retry every error?**  
Some errors are permanent — retrying wastes time and may add cost.

**Q8. How do you reduce API cost?**  
Measure tokens, reduce unnecessary context, choose appropriate models, constrain output, monitor usage.

**Q9. Why mock API calls in tests?**  
Faster, more predictable, independent of provider availability, credentials, and charges.

**Q10. Does a successful API response mean the answer is correct?**  
**No.** A successful HTTP response means the service returned output — correctness is a separate concern.

**Q11. Why isn't a prompt sufficient for authorization?**  
Prompts can be ignored or manipulated. **Application code** must enforce identity, permissions, and business rules.

**Q12. How would you design an app that can switch providers?**  
Define a **small internal interface**, isolate provider-specific handling, and test each adapter against the same contract.

---
## ✅ 18. Module Completion Checklist

- [ ] Explain **APIs, SDKs, and providers**
- [ ] Configure a provider **without hardcoding** credentials
- [ ] Send a **successful model request**
- [ ] Understand **message roles** and **response structure**
- [ ] Inspect **usage metadata** where available
- [ ] Implement / understand **streaming**
- [ ] Distinguish **sync vs async**
- [ ] Handle **timeouts and provider errors**
- [ ] Understand **retries, backoff, idempotency, rate limits**
- [ ] Estimate **costs** using applicable pricing
- [ ] Protect **credentials and sensitive data**
- [ ] Write **unit tests** with mocked responses
- [ ] Build the **reusable LLM client** project
- [ ] Commit the code and documentation to GitHub

---
## 🏁 19. Key Takeaways

- LLM APIs connect applications to **hosted or compatible model interfaces**
- SDKs **simplify** API usage — but HTTP & failure handling still matter
- **API keys belong in secure config**, never in committed source
- **Streaming & async** improve responsiveness and I/O handling
- **Retries must be bounded and selective**
- **Rate limits, token usage, latency, cost** are production concerns
- **Unit tests** shouldn't need a live model call for every run
- **Provider-specific behavior** should be **isolated** behind clear interfaces
- **Model output** requires **validation**, and sensitive actions require **app-side authorization**

### 🧭 Golden Rule

> **Make one request, understand the response, handle failure, test the integration, then build on it.**

---

### 🔗 What's Next?

**Module 04 — Prompt Engineering**

- Clear instructions & context
- Role & task definition
- **Zero-shot & few-shot** prompting
- Prompt **templates** and **variables**
- Summarization, classification, extraction
- **Task decomposition**
- **Prompt injection** awareness
- Prompt **versioning** & **evaluation**
- When prompting alone isn't enough